# kNN vs ScaNN — Search Mode Deep Dive

## The core question
> *If I delete the ScaNN index, does VS2.0 fall back to kNN automatically?*  
> **Yes — VS2.0 auto-routes: index present → ScaNN ANN, no index → exact kNN.**

## The problem with benchmarking directly
You can't run both modes on the same live collection at the same time —  
VS2.0 picks the mode based purely on whether an index exists.  
Deleting the index just to benchmark kNN is destructive and slow to rebuild.

## What this notebook does instead

| Section | Approach |
|---------|----------|
| 1 | Detect which mode is currently active |
| 2 | Benchmark **ScaNN ANN** latency (live API call) |
| 3 | Simulate **exact kNN in Python** — download vectors, compute dot products manually |
| 4 | **Compare results** — do ScaNN and kNN return the same top-K? |
| 5 | **Latency chart** — ScaNN vs Python-kNN side by side |
| 6 | **Scaling theory** — what happens at 10k, 100k, 1M DataObjects |
| 7 | Decision guide — when to use each mode |

---

```
ScaNN ANN  (index exists)          exact kNN  (no index)
──────────────────────────         ───────────────────────────
query → find nearest clusters      query → compare with EVERY
       → scan only that cluster           DataObject in full
       → return top-K                     → sort → return top-K

Recall : ~99%                      Recall : 100%
Latency: sub-10ms at any scale     Latency: O(N) — grows linearly
```

---
## 0 — Setup

In [ ]:
import sys
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import config

import vertexai
from google.cloud import vectorsearch_v1beta as vs
from vertexai.language_models import TextEmbeddingModel

vertexai.init(project=config.PROJECT_ID, location=config.LOCATION)

COLLECTION    = config.COLLECTION_RESOURCE
vs_client     = vs.VectorSearchServiceClient()
do_client     = vs.DataObjectServiceClient()
search_client = vs.DataObjectSearchServiceClient()
emb_model     = TextEmbeddingModel.from_pretrained(config.EMBEDDING_MODEL)

print(f"Collection : {config.COLLECTION_ID}")
print(f"Embedding  : {config.EMBEDDING_MODEL}")
print("Clients ready.")

---
## Section 1 — Detect Current Search Mode

VS2.0 auto-routes based purely on whether a Collection Index exists.  
No code change is needed when switching modes — the API call is identical.

In [ ]:
indexes = list(
    vs_client.list_indexes(
        request=vs.ListIndexesRequest(parent=COLLECTION)
    )
)

HAS_INDEX = len(indexes) > 0
ACTIVE_MODE = "ScaNN ANN" if HAS_INDEX else "exact kNN"

print("=" * 55)
print(f"  Indexes found : {len(indexes)}")
print(f"  Active mode   : {ACTIVE_MODE}")
if HAS_INDEX:
    idx = indexes[0]
    print(f"  Index ID      : {idx.name.split('/')[-1]}")
    print(f"  Created       : {idx.create_time}")
print("=" * 55)
print()
print("How VS2.0 decides which mode to use:")
print("  list_indexes() returns results  →  ScaNN ANN")
print("  list_indexes() returns empty    →  exact kNN (full scan)")
print()
print("The SearchDataObjectsRequest call is IDENTICAL in both modes.")
print("You never change any search code — VS2.0 handles routing internally.")

---
## Section 2 — Benchmark ScaNN ANN (Live API)

We run the same query multiple times and measure end-to-end latency.  
This is the **real production latency** with your current index.

In [ ]:
QUERY_TEXT = "cozy private room near downtown Austin under $100"
TOP_K      = 10
N_RUNS     = 15

# Embed query once (reuse for all runs)
[q_emb] = emb_model.get_embeddings([QUERY_TEXT])
q_vec   = list(q_emb.values)
print(f"Query : '{QUERY_TEXT}'")
print(f"Vector: {len(q_vec)}-dim")
print()

def scann_search(q_vec, top_k=TOP_K):
    """One SearchDataObjects call — returns (hits, elapsed_ms)."""
    req = vs.SearchDataObjectsRequest(
        parent        = COLLECTION,
        vector_search = vs.VectorSearch(
            vector       = vs.DenseVector(values=q_vec),
            search_field = "embedding",
            top_k        = top_k,
        ),
    )
    t0   = time.perf_counter()
    resp = search_client.search_data_objects(request=req)
    ms   = (time.perf_counter() - t0) * 1000
    hits = [(r.data_object.data_object_id, float(r.distance))
            for r in (resp.results or [])]
    return hits, ms

# Warm-up run (first call can be slower due to TCP setup)
_, _ = scann_search(q_vec)

# Benchmark
scann_latencies = []
scann_hits      = None
for i in range(N_RUNS):
    hits, ms = scann_search(q_vec)
    scann_latencies.append(ms)
    if i == 0:
        scann_hits = hits   # save first result for comparison later

print(f"ScaNN ANN — {N_RUNS} runs, top_k={TOP_K}:")
print(f"  Min    : {min(scann_latencies):.1f} ms")
print(f"  Max    : {max(scann_latencies):.1f} ms")
print(f"  Mean   : {np.mean(scann_latencies):.1f} ms")
print(f"  Median : {np.median(scann_latencies):.1f} ms")
print(f"  p95    : {np.percentile(scann_latencies, 95):.1f} ms")
print()
print(f"Top-{len(scann_hits)} hit IDs (first run):")
for rank, (obj_id, score) in enumerate(scann_hits, 1):
    print(f"  [{rank}] {obj_id[:20]}...  score={score:.4f}")

---
## Section 3 — Simulate Exact kNN in Python

Since we can't delete the live index just to benchmark kNN, we **reproduce kNN manually**:

1. Download a sample of DataObjects (with their stored 768-dim vectors)
2. Load them all into a NumPy matrix
3. Compute `query_vector @ all_vectors.T` — this is the exact dot-product scan kNN does
4. Sort and pick top-K

This is **exactly what VS2.0 does internally when no index exists** — a full brute-force scan.

In [ ]:
# Step 3a — Download DataObjects with their embedding vectors
# We page through query_data_objects and collect all vectors.
# For 3,000 listings this takes ~30–60 seconds.

MAX_OBJECTS = 3000   # cap — increase if you ingested more
PAGE_SIZE   = 200

print(f"Downloading up to {MAX_OBJECTS} DataObjects with vectors...")
print("(This simulates what kNN must load into memory at query time)")
print()

all_ids     = []
all_vectors = []
page_token  = None

t0 = time.time()
while len(all_ids) < MAX_OBJECTS:
    req = vs.QueryDataObjectsRequest(
        parent        = COLLECTION,
        page_size     = PAGE_SIZE,
        page_token    = page_token or "",
        output_fields = vs.OutputFields(
            vector_fields = ["embedding"],   # request the stored vectors
        ),
    )
    resp = search_client.query_data_objects(request=req)
    for obj in resp.data_objects:
        if "embedding" in obj.vectors:
            all_ids.append(obj.data_object_id)
            all_vectors.append(list(obj.vectors["embedding"].dense.values))
    page_token = resp.next_page_token
    print(f"  Downloaded {len(all_ids)} so far...", end="\r")
    if not page_token or len(all_ids) >= MAX_OBJECTS:
        break

elapsed = time.time() - t0
V = np.array(all_vectors, dtype=np.float32)   # shape: (N, 768)
print(f"\n✓ Downloaded {len(all_ids)} DataObjects in {elapsed:.1f}s")
print(f"  Vector matrix shape : {V.shape}")
print(f"  Memory (float32)    : {V.nbytes / 1024 / 1024:.1f} MB")

In [ ]:
# Step 3b — Python-side exact kNN using NumPy dot product
# This is the actual computation VS2.0 does internally during a kNN scan.

def python_knn(q_vec, vector_matrix, ids, top_k=TOP_K):
    """
    Exact kNN via brute-force dot-product scan.

    Steps:
      1. q @ V.T  →  dot-product score for every vector in the matrix
      2. argsort descending  →  ranked indices
      3. return top-K (id, score) pairs
    """
    q  = np.array(q_vec, dtype=np.float32)          # (768,)
    scores = vector_matrix @ q                       # (N,)  — one dot product per row
    top_indices = np.argsort(scores)[::-1][:top_k]  # descending sort, take top-K
    return [(ids[i], float(scores[i])) for i in top_indices]

# Warm up NumPy (JIT-like first-call overhead)
_ = python_knn(q_vec, V, all_ids)

# Benchmark exact kNN
N_RUNS_KNN = 50   # more runs because it's fast in NumPy
knn_latencies = []
knn_hits      = None

for i in range(N_RUNS_KNN):
    t0   = time.perf_counter()
    hits = python_knn(q_vec, V, all_ids)
    ms   = (time.perf_counter() - t0) * 1000
    knn_latencies.append(ms)
    if i == 0:
        knn_hits = hits

print(f"Python exact kNN — {N_RUNS_KNN} runs, top_k={TOP_K}, N={len(all_ids)} vectors:")
print(f"  Min    : {min(knn_latencies):.2f} ms")
print(f"  Max    : {max(knn_latencies):.2f} ms")
print(f"  Mean   : {np.mean(knn_latencies):.2f} ms")
print(f"  Median : {np.median(knn_latencies):.2f} ms")
print(f"  p95    : {np.percentile(knn_latencies, 95):.2f} ms")
print()
print(f"NOTE: This is Python + NumPy kNN — raw compute only.")
print(f"Actual VS2.0 kNN also includes gRPC network overhead (add ~10–30ms).")

---
## Section 4 — Compare Results: Do ScaNN and kNN Agree?

ScaNN is **approximate** — it may miss some true nearest neighbors. At ~99% recall,  
for a top-10 query you'd expect 9–10 of the results to match exact kNN.

In [ ]:
# Compare ScaNN results vs exact kNN results
scann_ids = [obj_id for obj_id, _ in scann_hits]
knn_ids   = [obj_id for obj_id, _ in knn_hits]

# Overlap at top-K
overlap = len(set(scann_ids) & set(knn_ids))
recall_at_k = overlap / TOP_K * 100

print(f"=== Result Comparison: ScaNN vs exact kNN (top-{TOP_K}) ===")
print()
print(f"  {'Rank':<6}  {'ScaNN ID (first 16)':<20}  {'Score':>8}  "
      f"{'kNN ID (first 16)':<20}  {'Score':>8}  {'Match?'}")
print("  " + "-" * 80)

for rank in range(TOP_K):
    s_id, s_score = scann_hits[rank] if rank < len(scann_hits) else ("-", 0)
    k_id, k_score = knn_hits[rank]   if rank < len(knn_hits)   else ("-", 0)
    match = "✓" if s_id == k_id else "≠"
    print(f"  {rank+1:<6}  {str(s_id)[:18]:<20}  {s_score:>8.4f}  "
          f"{str(k_id)[:18]:<20}  {k_score:>8.4f}  {match}")

print()
print(f"  IDs in common : {overlap} / {TOP_K}")
print(f"  Recall@{TOP_K}    : {recall_at_k:.0f}%")
print()
if recall_at_k >= 90:
    print("  ✓ ScaNN recall is high — the 1% approximation is acceptable for RAG.")
else:
    print("  ⚠ Lower recall than expected — this collection may be too small for ScaNN")
    print("    to fully warm up its cluster structure. Recall improves at 100k+ objects.")

In [ ]:
# Fetch names for both result sets to make comparison human-readable
def fetch_name(obj_id):
    try:
        obj = do_client.get_data_object(
            request=vs.GetDataObjectRequest(
                name=f"{COLLECTION}/dataObjects/{obj_id}"
            )
        )
        return dict(obj.data).get("name", obj_id) if obj.data else obj_id
    except Exception:
        return obj_id

print("Fetching listing names for top-5 from each mode...")
print()
print(f"  {'Rank':<5}  {'ScaNN top-5':<42}  {'kNN top-5':<42}")
print("  " + "-" * 90)
for rank in range(min(5, TOP_K)):
    s_name = fetch_name(scann_hits[rank][0])[:40] if rank < len(scann_hits) else "-"
    k_name = fetch_name(knn_hits[rank][0])[:40]   if rank < len(knn_hits)   else "-"
    match  = "✓" if scann_hits[rank][0] == knn_hits[rank][0] else "≠"
    print(f"  {rank+1:<5}  {s_name:<42}  {k_name:<42}  {match}")

---
## Section 5 — Latency Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: run-by-run latency ─────────────────────────────────────────────────
ax = axes[0]
x_scann = range(1, len(scann_latencies) + 1)
x_knn   = range(1, len(knn_latencies)   + 1)

ax.plot(x_scann, scann_latencies, "o-", color="#FF385C", markersize=5,
        label=f"ScaNN ANN  (mean={np.mean(scann_latencies):.1f}ms)", linewidth=1.5)
ax.axhline(np.mean(scann_latencies), color="#FF385C", linestyle="--", alpha=0.5)
ax.axhline(np.mean(knn_latencies),   color="#4285F4", linestyle="--", alpha=0.5)

# kNN runs on a separate scale since it's much faster at 3k items
ax2 = ax.twinx()
ax2.plot(x_knn[:N_RUNS], knn_latencies[:N_RUNS], "s-", color="#4285F4", markersize=4,
         label=f"Python kNN  (mean={np.mean(knn_latencies):.2f}ms)", linewidth=1.5, alpha=0.8)
ax2.set_ylabel("Python kNN latency (ms)", color="#4285F4")
ax2.tick_params(axis="y", labelcolor="#4285F4")

ax.set_xlabel("Run #")
ax.set_ylabel("ScaNN latency (ms)", color="#FF385C")
ax.tick_params(axis="y", labelcolor="#FF385C")
ax.set_title(f"Latency per run — {len(all_ids):,} DataObjects, top-{TOP_K}")

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)

# ── Right: summary bar chart ─────────────────────────────────────────────────
ax = axes[1]
modes  = [f"ScaNN ANN\n({len(all_ids):,} vectors)", f"Python kNN\n({len(all_ids):,} vectors)"]
means  = [np.mean(scann_latencies), np.mean(knn_latencies)]
p95s   = [np.percentile(scann_latencies, 95), np.percentile(knn_latencies, 95)]
colors = ["#FF385C", "#4285F4"]

x   = np.arange(2)
w   = 0.35
b1  = ax.bar(x - w/2, means, w, label="Mean",  color=colors, alpha=0.85, edgecolor="white")
b2  = ax.bar(x + w/2, p95s,  w, label="p95",   color=colors, alpha=0.45, edgecolor="white", hatch="//")

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}ms", ha="center", va="bottom", fontsize=9)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}ms", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_ylabel("Latency (ms)")
ax.set_title("Mean vs p95 latency")
ax.legend()

note = ("NOTE: Python kNN is raw compute only.\n"
        "Real VS2.0 kNN adds ~10–30ms gRPC overhead.")
fig.text(0.5, -0.04, note, ha="center", fontsize=8.5, color="#666")

plt.suptitle(f"kNN vs ScaNN — Latency Comparison  |  {len(all_ids):,} DataObjects  |  top-{TOP_K}",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 6 — Scaling Theory: What Happens at 10k / 100k / 1M Objects?

At 3,000 DataObjects, kNN is fast even in Python. The gap becomes critical at scale.

In [ ]:
# Extrapolate kNN latency based on O(N) assumption
# and contrast with ScaNN which stays roughly flat

N_measured   = len(all_ids)
knn_mean_ms  = np.mean(knn_latencies)   # measured at N_measured
# Estimate VS2.0 kNN including gRPC overhead
knn_grpc_overhead = 15.0  # ms — typical for a local/regional VS2.0 call
knn_base_ms  = knn_mean_ms + knn_grpc_overhead

# kNN scales O(N) — linear extrapolation
Ns = np.array([1_000, 3_000, 10_000, 50_000, 100_000, 500_000, 1_000_000])
knn_ms   = knn_base_ms * (Ns / N_measured)   # linear scaling

# ScaNN latency stays roughly constant (O(log N) approximated as flat for practical purposes)
scann_base   = np.mean(scann_latencies)
scann_ms     = np.full_like(Ns, scann_base, dtype=float)
# slight growth at very large scale
scann_ms[Ns > 100_000] *= 1.5
scann_ms[Ns > 500_000] *= 2.0

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(Ns, knn_ms,   "o-", color="#4285F4", linewidth=2, markersize=7, label="exact kNN (O(N))")
ax.plot(Ns, scann_ms, "s-", color="#FF385C", linewidth=2, markersize=7, label="ScaNN ANN (~O(log N))")

# Mark current collection size
ax.axvline(N_measured, color="#888", linestyle=":", linewidth=1.5)
ax.text(N_measured * 1.05, ax.get_ylim()[1] * 0.9,
        f"Current:\n{N_measured:,} objects", fontsize=8.5, color="#555")

# Threshold where ScaNN pays off
crossover = Ns[np.argmax(knn_ms > scann_ms * 2)]
ax.axvspan(crossover, Ns[-1], alpha=0.07, color="#FF385C",
           label="ScaNN advantage zone")

ax.set_xscale("log")
ax.set_xlabel("Number of DataObjects (log scale)")
ax.set_ylabel("Search latency (ms)")
ax.set_title("Projected Search Latency: kNN vs ScaNN as Collection Grows")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Annotations at scale points
for N, km, sm in zip(Ns, knn_ms, scann_ms):
    if N in [10_000, 100_000, 1_000_000]:
        ax.annotate(f"{km:.0f}ms", xy=(N, km), xytext=(N*1.1, km*1.1),
                    fontsize=7.5, color="#4285F4")
        ax.annotate(f"{sm:.0f}ms", xy=(N, sm), xytext=(N*1.1, sm*0.7),
                    fontsize=7.5, color="#FF385C")

plt.tight_layout()
plt.show()

print("Projected latency table:")
print(f"  {'Objects':<12}  {'kNN (ms)':>10}  {'ScaNN (ms)':>12}  {'ScaNN speedup':>14}")
print("  " + "-" * 55)
for N, km, sm in zip(Ns, knn_ms, scann_ms):
    speedup = km / sm
    marker = "  ← current" if N == N_measured else ""
    print(f"  {N:<12,}  {km:>10.1f}  {sm:>12.1f}  {speedup:>13.1f}x{marker}")

---
## Section 7 — Decision Guide: Which Mode to Use?

In [ ]:
# Visual decision flowchart
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)

def box(ax, x, y, w, h, text, color, face=None, fontsize=9):
    face = face or color + "22"
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.1", lw=1.8, edgecolor=color, facecolor=face))
    ax.text(x+w/2, y+h/2, text, ha="center", va="center",
            fontsize=fontsize, color=color, fontweight="bold", multialignment="center")

def arr(ax, x1, y1, x2, y2, label="", color="#888"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
        arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
    if label:
        ax.text((x1+x2)/2+0.1, (y1+y2)/2, label, fontsize=8, color="#555")

# Decision diamond
box(ax, 4.5, 4.2, 3.0, 1.2, "Collection\nsize?", "#555", "#f5f5f5", fontsize=10)

# kNN path
box(ax, 0.3, 2.0, 3.5, 1.4,
    "exact kNN\n(no index needed)", "#4285F4")
box(ax, 0.3, 0.2, 3.5, 1.5,
    "✓ 100% recall\n✓ instant setup\n✓ zero build cost\n✓ fine for < 100k objects",
    "#4285F4", "#e8f0fe", fontsize=8)

# ScaNN path
box(ax, 8.2, 2.0, 3.5, 1.4,
    "ScaNN ANN\n(create index)", "#FF385C")
box(ax, 8.2, 0.2, 3.5, 1.5,
    "✓ ~99% recall\n✓ sub-10ms at any scale\n✓ required for production\n⚠ 5–20 min build time",
    "#FF385C", "#fde8ec", fontsize=8)

# Common box
box(ax, 4.2, 2.0, 3.6, 1.4,
    "SearchDataObjects\nAPI call is IDENTICAL\nin both modes", "#34A853")

arr(ax, 4.5, 4.2, 2.0, 3.4, "< 100k", "#4285F4")
arr(ax, 7.5, 4.2, 9.9, 3.4, "> 100k", "#FF385C")
arr(ax, 6.0, 4.2, 6.0, 3.4, color="#34A853")

arr(ax, 2.0, 2.0, 2.0, 1.7, color="#4285F4")
arr(ax, 9.9, 2.0, 9.9, 1.7, color="#FF385C")

ax.set_title("VS2.0 Search Mode Decision Guide", fontsize=13, pad=10)
ax.text(6.0, 5.6, f"Current collection: {len(all_ids):,} DataObjects  →  "
        f"{'ScaNN ANN (index active)' if HAS_INDEX else 'exact kNN (no index)'}",
        ha="center", fontsize=9.5, color="#333")
plt.tight_layout()
plt.show()

In [ ]:
# Final summary printout
print("=" * 60)
print("  BENCHMARK SUMMARY")
print("=" * 60)
print(f"  Collection      : {config.COLLECTION_ID}")
print(f"  DataObjects     : {len(all_ids):,}")
print(f"  Query           : '{QUERY_TEXT[:45]}...'")
print(f"  top_k           : {TOP_K}")
print()
print(f"  ScaNN ANN (live API)")
print(f"    Mean latency  : {np.mean(scann_latencies):.1f} ms")
print(f"    p95  latency  : {np.percentile(scann_latencies, 95):.1f} ms")
print(f"    Recall@{TOP_K}    : {recall_at_k:.0f}%")
print()
print(f"  Python kNN (simulated, no gRPC)")
print(f"    Mean latency  : {np.mean(knn_latencies):.2f} ms  (pure compute)")
print(f"    + gRPC est.   : ~{np.mean(knn_latencies) + knn_grpc_overhead:.1f} ms  (realistic VS2.0 kNN)")
print(f"    Recall@{TOP_K}    : 100% (exact)")
print()
print(f"  At this scale ({len(all_ids):,} objects):")
if HAS_INDEX:
    print(f"    → ScaNN index is active. For {len(all_ids):,} objects, kNN would also")
    print(f"      be fast — but ScaNN is already there so no reason to remove it.")
    print(f"    → At 100k+ objects the ScaNN advantage becomes critical.")
else:
    print(f"    → No index — running exact kNN. Fine at this scale.")
    print(f"    → Run scripts/03_create_index.py when you scale beyond 100k objects.")
print("=" * 60)